In [1]:

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import hvplot.pandas
from sklearn.decomposition import PCA
import chardet

Python Script to analyze Superstore sales data for comparison to Tableau & R

In [11]:
# Checking encoding type of our csv file using chardet (encoding is not UTF-8)

with open("Superstore_Sales.csv", 'rb') as f:
    result = chardet.detect(f.read())

print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


In [12]:
# Read the CSV file using the proper encoding (thanks chardet!)
superstore_df = pd.read_csv("Superstore_Sales.csv", encoding='Windows-1252')

# Display DataFrame content
print(superstore_df)

      Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
0          1  CA-2016-152156   11/8/2016  11/11/2016    Second Class   
1          2  CA-2016-152156   11/8/2016  11/11/2016    Second Class   
2          3  CA-2016-138688   6/12/2016   6/16/2016    Second Class   
3          4  US-2015-108966  10/11/2015  10/18/2015  Standard Class   
4          5  US-2015-108966  10/11/2015  10/18/2015  Standard Class   
...      ...             ...         ...         ...             ...   
9989    9990  CA-2014-110422   1/21/2014   1/23/2014    Second Class   
9990    9991  CA-2017-121258   2/26/2017    3/3/2017  Standard Class   
9991    9992  CA-2017-121258   2/26/2017    3/3/2017  Standard Class   
9992    9993  CA-2017-121258   2/26/2017    3/3/2017  Standard Class   
9993    9994  CA-2017-119914    5/4/2017    5/9/2017    Second Class   

     Customer ID     Customer Name    Segment        Country             City  \
0       CG-12520       Claire Gute   Consumer  United 

In [13]:
# Preliminary formatting of data types, for creating our calculated fields

# 'Profit' and 'Sales' columns to numeric
superstore_df['Profit'] = pd.to_numeric(superstore_df['Profit'], errors='coerce')
superstore_df['Sales'] = pd.to_numeric(superstore_df['Sales'], errors='coerce')

# Calculate the Profit Ratio:
# Divide profit by sales
superstore_df['Profit Ratio'] = superstore_df['Profit'] / superstore_df['Sales']

# Handle NaNs & nulls (division by zero errors) 
superstore_df['Profit Ratio'].replace([float('inf'), -float('inf')], pd.NA, inplace=True)

# DF check
superstore_df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit Ratio
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,0.1600
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,0.3000
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,0.4700
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,-0.4000
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,0.1125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,1/21/2014,1/23/2014,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028,0.1625
9990,9991,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332,0.1700
9991,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932,0.0750
9992,9993,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200,0.4500


In [14]:
# 'Order Date' and 'Ship Date' to datetime
superstore_df['Order Date'] = pd.to_datetime(superstore_df['Order Date'], format='%m/%d/%Y')
superstore_df['Ship Date'] = pd.to_datetime(superstore_df['Ship Date'], format='%m/%d/%Y')

# Calculate Lead Time as the difference between 'Ship Date' and 'Order Date'
superstore_df['Lead Time'] = (superstore_df['Ship Date'] - superstore_df['Order Date']).dt.days
superstore_df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit Ratio,Lead Time
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,0.1600,3
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,0.3000,3
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,0.4700,4
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,-0.4000,7
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,0.1125,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028,0.1625,2
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332,0.1700,5
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932,0.0750,5
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200,0.4500,5


In [9]:
# Optional installation for mapping libraries

#pip install geopandas matplotlib folium


Note: you may need to restart the kernel to use updated packages.


Now, we create versions of our Tableau visualizations using Python tools. Note- Style and aesthetics are simple in these but can be adjusted in parameters and/or more advanced libraries

In [18]:
# Map imports

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px


Initial mapping of our Profit Ratio by State

In [20]:
# Aggregate average profit ratio by state
state_profit_ratio = superstore_df.groupby('State')['Profit Ratio'].mean().reset_index()

# Define custom color scale
custom_color_scale = [
    [0, 'red'],       # Lowest values in red
    [0.5, 'lightgreen'],  # Mid values in light green
    [1, 'darkgreen']  # Highest values in dark green
]

# Create an interactive map
fig = px.choropleth(
    state_profit_ratio,
    geojson="https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
    locations='State',
    featureidkey="properties.name",  # Adjust based on the GeoJSON file
    color='Profit Ratio',
    color_continuous_scale=custom_color_scale,  # Use the custom color scale
    scope="usa",
    labels={'Profit Ratio': 'Average Profit Ratio'}
)

fig.update_geos(fitbounds="locations")
fig.update_layout(title_text="Average Profit Ratio by State")
fig.show()

This is a simpler version of our Tableau map. We lack some of the interactive features available with Tableau, but using Dash allows us to add sliders.

In [21]:
# Optional Dash installation

#pip install dash


  Obtaining dependency information for dash from https://files.pythonhosted.org/packages/27/ad/7047095224013ec2ae37ba8ece5956773e7953c39a3af5aa20d821ed99aa/dash-2.17.1-py3-none-any.whl.metadata
  Obtaining dependency information for dash-html-components==2.0.0 from https://files.pythonhosted.org/packages/75/65/1b16b853844ef59b2742a7de74a598f376ac0ab581f0dcc34db294e5c90e/dash_html_components-2.0.0-py3-none-any.whl.metadata
  Obtaining dependency information for dash-core-components==2.0.0 from https://files.pythonhosted.org/packages/00/9e/a29f726e84e531a36d56cff187e61d8c96d2cc253c5bcef9a7695acb7e6a/dash_core_components-2.0.0-py3-none-any.whl.metadata
  Obtaining dependency information for dash-table==5.0.0 from https://files.pythonhosted.org/packages/da/ce/43f77dc8e7bbad02a9f88d07bf794eaf68359df756a28bb9f2f78e255bb1/dash_table-5.0.0-py3-none-any.whl.metadata
  Obtaining dependency information for retrying from https://files.pythonhosted.org/packages/8f/04/9e36f28be4c0532c0e9207ff9dc01fb

In [22]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import geopandas as gpd

# Load your data
superstore_df = pd.read_csv("Superstore_Sales.csv", encoding='Windows-1252')
superstore_df['Profit Ratio'] = superstore_df['Profit'] / superstore_df['Sales']
superstore_df['Order Date'] = pd.to_datetime(superstore_df['Order Date'])

# Load US states GeoJSON
us_states_geojson = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"

# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout
app.layout = html.Div([
    dcc.Graph(id='map'),
    dcc.RangeSlider(
        id='date-slider',
        min=superstore_df['Order Date'].min().timestamp() * 1000,  # milliseconds
        max=superstore_df['Order Date'].max().timestamp() * 1000,  # milliseconds
        marks={int(superstore_df['Order Date'].min().timestamp() * 1000): superstore_df['Order Date'].min().strftime('%Y-%m-%d'),
               int(superstore_df['Order Date'].max().timestamp() * 1000): superstore_df['Order Date'].max().strftime('%Y-%m-%d')},
        step=24*60*60*1000,  # step size in milliseconds (1 day)
        value=[int(superstore_df['Order Date'].min().timestamp() * 1000), int(superstore_df['Order Date'].max().timestamp() * 1000)]
    )
])

# Define callback to update map based on slider input
@app.callback(
    Output('map', 'figure'),
    [Input('date-slider', 'value')]
)
def update_map(date_range):
    start_date = pd.to_datetime(date_range[0], unit='ms')
    end_date = pd.to_datetime(date_range[1], unit='ms')

    # Filter data based on date range
    filtered_df = superstore_df[(superstore_df['Order Date'] >= start_date) & (superstore_df['Order Date'] <= end_date)]

    # Aggregate average profit ratio by state
    state_profit_ratio = filtered_df.groupby('State')['Profit Ratio'].mean().reset_index()

    # Create the map
    fig = px.choropleth(
        state_profit_ratio,
        geojson=us_states_geojson,
        locations='State',
        featureidkey="properties.name",
        color='Profit Ratio',
        color_continuous_scale=['red', 'lightgreen', 'darkgreen'],
        scope="usa",
        labels={'Profit Ratio': 'Average Profit Ratio'}
    )

    fig.update_geos(fitbounds="locations")
    fig.update_layout(title_text="Average Profit Ratio by State")
    return fig

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)


Now we can add filter sliders, like date ranges, similar to Tableau's presentation. More effort and coding knowledge is required in comparison.

In [16]:
# Check to ensure our original DF with added columns appear

print(superstore_df.columns)

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit',
       'Profit Ratio', 'Lead Time'],
      dtype='object')
